# GISPR Module 5 — Introduction to Raster Data

**Course:** GIS Spatial Analysis with Python and R (GISPR)  
**Module:** 5 — Raster Data: Arrays, Metadata, Statistics, and Visualisation  
**Builds on:** Module 4 (vector operations — buffer, clip, dissolve, spatial join)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Explain how every raster maps to a NumPy array or R matrix — shape, dtype, NoData, affine transform | Section 1 |
| 2 | Load float, integer, and categorical rasters using `rasterio`, `terra`, `raster`, and `stars` | Section 2 |
| 3 | Run the seven-point first-inspection checklist on any GeoTIFF in both languages | Section 3 |
| 4 | Index arrays, extract cell values, and use windowed reads for large files | Section 4 |
| 5 | Compute correct summary statistics — with NoData masking — for continuous and categorical rasters | Section 5 |
| 6 | Visualise single-band maps, histograms, and multi-band RGB composites | Section 6 |
| 7 | Wrap the inspection workflow in a reusable `inspect_raster()` function that carries into Module 6 | Section 7 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste into your terminal / Anaconda Prompt, **not** in a notebook cell

> **The framing problem for this module:**  
> You receive two rasters from a colleague — a 30 m DEM and an NLCD land cover grid covering the same watershed. Before running any terrain or habitat analysis you need to know: Are the files healthy? Do they share a CRS? What are the plausible value ranges? Are there NoData holes? Do not touch summary statistics until you can answer all of those questions.  
> That is the inspection checklist you will build here — and it becomes the entry point for every raster operation in Modules 6 and 7.

### Data in this notebook
All rasters are generated synthetically — no downloads required, every cell runs identically for every learner.

| File | Dtype | Represents | Section used |
|---|---|---|---|
| `output/dem_30m.tif` | `float32` | 30 m DEM, Mt. Rainier foothills study area | 1 – 7 |
| `output/nlcd_cover.tif` | `uint8` | NLCD land cover class codes (integer categorical) | 2, 3, 5, 6 |
| `output/landsat8.tif` | `uint16` | 7-band Landsat 8 scaled reflectance | 2, 5, 6 |


### Setup — install packages and create synthetic rasters

Run this cell **once** before any other cell. It installs missing packages and writes the three GeoTIFFs every subsequent section reads.


In [ ]:
# [Terminal] Install Python dependencies if not already present
# pip install rasterio numpy matplotlib
#
# [Terminal] Install R packages if not already present — run once in an R session:
# install.packages(c("terra", "stars", "raster", "ggplot2", "viridis"))


In [ ]:
# [Python] Create all synthetic rasters used throughout this notebook.
# Rerun this cell if you delete the output/ folder.
#
# Study area: ~10 km × 10 km tile, UTM Zone 10N (EPSG:32610)
# Origin: near Enumclaw WA — gateway to Mt. Rainier

import numpy as np
import rasterio
from rasterio.transform import from_origin
from pathlib import Path

Path("output").mkdir(exist_ok=True)

# ── Shared spatial parameters ─────────────────────────────────────────────
EPSG   = 32610
WEST   = 620000.0    # easting,  metres
NORTH  = 5230000.0   # northing, metres
RES    = 30.0        # 30 m pixels
NROWS  = 334         # 334 × 30 m ≈ 10 km
NCOLS  = 334
rng    = np.random.default_rng(42)   # reproducible seed

transform = from_origin(WEST, NORTH, RES, RES)

# ── 1. DEM (float32) ─────────────────────────────────────────────────────
# Elevation rises SW → NE; Gaussian noise gives terrain texture.
# A NoData hole in the NE corner simulates a cloud mask or data gap.
row_idx, col_idx = np.mgrid[0:NROWS, 0:NCOLS]
dem = (400.0
       + (row_idx / NROWS) * 800.0
       + (col_idx / NCOLS) * 400.0
       + rng.normal(0, 25, (NROWS, NCOLS))).astype("float32")

NODATA_DEM = -9999.0
dem[0:30, 300:334] = NODATA_DEM            # punch a NoData hole

with rasterio.open(
    "output/dem_30m.tif", "w", driver="GTiff",
    height=NROWS, width=NCOLS, count=1, dtype="float32",
    crs=f"EPSG:{EPSG}", transform=transform, nodata=NODATA_DEM
) as dst:
    dst.write(dem, 1)
print("✓  output/dem_30m.tif")

# ── 2. NLCD land cover (uint8 categorical) ───────────────────────────────
# Low elevation → open water / developed; high elevation → forest / shrub.
# NLCD class codes: 11=Water, 21=Dev, 41=Decid, 42=Evergr, 52=Shrub,
#                   71=Grass, 81=Pasture, 90=WoodyWetland
NLCD_CODES = np.array([11, 21, 41, 42, 52, 71, 81, 90], dtype="uint8")
elev_norm  = np.clip((dem - 400) / 1200, 0, 1)
class_idx  = (elev_norm * (len(NLCD_CODES) - 1)).astype(int)
nlcd       = NLCD_CODES[class_idx].astype("uint8")
nlcd[dem == NODATA_DEM] = 0                # NoData → 0 (background)

with rasterio.open(
    "output/nlcd_cover.tif", "w", driver="GTiff",
    height=NROWS, width=NCOLS, count=1, dtype="uint8",
    crs=f"EPSG:{EPSG}", transform=transform, nodata=0
) as dst:
    dst.write(nlcd, 1)
print("✓  output/nlcd_cover.tif")

# ── 3. Landsat 8 — 7 bands (uint16, scaled reflectance × 10 000) ─────────
# Band order: B1 Coastal, B2 Blue, B3 Green, B4 Red,
#             B5 NIR, B6 SWIR1, B7 SWIR2
BAND_MEANS = [400, 600, 700, 650, 3200, 1800, 1100]
ls_bands   = np.stack([
    np.clip(rng.normal(mu, mu * 0.15, (NROWS, NCOLS)), 0, 10000).astype("uint16")
    for mu in BAND_MEANS
])

with rasterio.open(
    "output/landsat8.tif", "w", driver="GTiff",
    height=NROWS, width=NCOLS, count=7, dtype="uint16",
    crs=f"EPSG:{EPSG}", transform=transform, nodata=0
) as dst:
    dst.write(ls_bands)
print("✓  output/landsat8.tif")

print(f"\nAll three rasters written.  Shape: {NROWS}×{NCOLS}  CRS: EPSG:{EPSG}")


---
## Section 1 — Arrays and Matrices: The Data Structure Behind Every Raster

A raster is a regular grid of cells. Each cell stores one numeric value per band. When you open a GeoTIFF in code, what you get is:
- **Python:** a NumPy `ndarray` — the same object you use for any numerical computation
- **R:** a numeric matrix (single band) or 3-D array (multi-band)

The **metadata** — CRS, extent, cell size, NoData value, affine transform — gives those numbers geographic meaning. Strip the metadata and you have an ordinary array of numbers.

### Shape conventions

| Context | Python shape | R dim |
|---|---|---|
| Single-band raster | `(rows, cols)` | `(rows, cols)` |
| Multi-band raster | `(bands, rows, cols)` | `(rows, cols, bands)` |
| Geographic origin | Row 0, Col 0 = **NW corner** | Row 1, Col 1 = **NW corner** |

> ⚠️ **Row 0 is NORTH, not south.** Array row index increases *downward* (southward). Geographic y increases *upward* (northward). They are opposites. The affine transform stored in the raster converts between them. Forgetting this is the most common raster indexing mistake.

### Raster dtype — what kind of numbers each cell holds

| Dtype | Range | Typical use | Caution |
|---|---|---|---|
| `uint8` | 0 – 255 | Land cover codes, 8-bit imagery | Arithmetic wraps: 200 + 100 = 44 |
| `int16` | −32 768 – 32 767 | Elevation with negatives, temperature | Wraps on overflow |
| `uint16` | 0 – 65 535 | 16-bit satellite imagery | Cast to float before math |
| `float32` | ±3.4 × 10³⁸ | DEM elevation, analysis outputs | Safe for arithmetic |
| `float64` | ±1.7 × 10³⁰⁸ | High-precision science | Large memory footprint |

**Rule:** cast to `float32` before any arithmetic on integer rasters.


### 1a — NumPy arrays in Python (the engine behind rasterio)


In [ ]:
# [Python] NumPy arrays — everything rasterio reads lands here

import numpy as np

# ── 1. Create arrays the same shape as real rasters ───────────────────────
# Single-band float32 — like a small DEM
dem_small = np.array([
    [412.3, 418.7, 431.1, 445.8],
    [420.1, 429.5, 441.0, 458.3],
    [435.6, 448.2, 460.9, 475.1],
    [449.8, 462.3, 477.4, 492.7]
], dtype="float32")

print("Single-band array (4×4 DEM-like):")
print(dem_small)
print(f"  shape:  {dem_small.shape}   → (rows={dem_small.shape[0]}, cols={dem_small.shape[1]})")
print(f"  dtype:  {dem_small.dtype}")
print(f"  size:   {dem_small.size} cells  |  nbytes: {dem_small.nbytes} B")

# Multi-band uint16 — like 3 Landsat bands
rgb_small = np.zeros((3, 4, 4), dtype="uint16")   # (bands, rows, cols)
rgb_small[0] = 650    # Red band
rgb_small[1] = 700    # Green band
rgb_small[2] = 3200   # NIR band (much higher — vegetation reflects NIR strongly)
print(f"\nMulti-band array:  shape={rgb_small.shape}  → (bands={rgb_small.shape[0]}, rows={rgb_small.shape[1]}, cols={rgb_small.shape[2]})")

# ── 2. Indexing — [row, col]; row 0 = NW (NORTH) ──────────────────────────
print("\nCorner indexing (row 0 = NORTH edge):")
print(f"  NW [0,  0 ]: {dem_small[0,  0]:.1f} m")
print(f"  NE [0, -1 ]: {dem_small[0, -1]:.1f} m")
print(f"  SW [-1, 0 ]: {dem_small[-1, 0]:.1f} m")
print(f"  SE [-1, -1]: {dem_small[-1, -1]:.1f} m")
print(f"  Centre [1:3, 1:3]:\n{dem_small[1:3, 1:3]}")

# Multi-band: [band, row, col]
print(f"\n  NIR band 2, centre [2, 1, 2]: {rgb_small[2, 1, 2]}")

# ── 3. Vectorised math — no for-loops ─────────────────────────────────────
dem_centred = dem_small - dem_small.mean()
print(f"\nCentred (subtract mean {dem_small.mean():.1f} m):")
print(dem_centred.round(1))

# NDVI: (NIR − Red) / (NIR + Red) — element-wise, no loop
nir = rgb_small[2].astype("float32")
red = rgb_small[0].astype("float32")
ndvi = (nir - red) / (nir + red)
print(f"\nNDVI (NIR−Red)/(NIR+Red)  range: [{ndvi.min():.3f}, {ndvi.max():.3f}]")

# ── 4. Dtype cast — ALWAYS before arithmetic on integers ──────────────────
nlcd_raw = np.array([11, 21, 41, 42], dtype="uint8")
print(f"\nuint8 overflow demo: 200 + 100 = {np.uint8(200) + np.uint8(100)}  (wraps!)")
print(f"Safe cast first:      200 + 100 = {np.float32(200) + np.float32(100)}")


### 1b — Matrices and arrays in R (the engine behind terra)


In [ ]:
# [R] Matrices and 3-D arrays — what terra and raster store internally

# ── 1. Single-band matrix ─────────────────────────────────────────────────
dem_small <- matrix(
    c(412.3, 418.7, 431.1, 445.8,
      420.1, 429.5, 441.0, 458.3,
      435.6, 448.2, 460.9, 475.1,
      449.8, 462.3, 477.4, 492.7),
    nrow = 4, ncol = 4, byrow = TRUE
)
storage.mode(dem_small) <- "double"   # float64

cat("Single-band matrix (4×4 DEM-like):\n")
print(dem_small)
cat(sprintf("  class: %s  dims: %d×%d  storage: %s\n",
    class(dem_small), nrow(dem_small), ncol(dem_small), storage.mode(dem_small)))

# Multi-band: R uses [row, col, band] — opposite of NumPy's [band, row, col]
rgb_small <- array(0L, dim = c(4, 4, 3))
rgb_small[,,1] <- 650L    # Red
rgb_small[,,2] <- 700L    # Green
rgb_small[,,3] <- 3200L   # NIR
storage.mode(rgb_small) <- "integer"
cat(sprintf("\nMulti-band array:  dim = [%s]  (rows × cols × bands)\n",
    paste(dim(rgb_small), collapse = " × ")))

# ── 2. Indexing — R is 1-indexed; [1, 1] = NW corner ─────────────────────
cat("\nCorner indexing (row 1 = NORTH edge):\n")
cat(sprintf("  NW [1, 1]:         %.1f m\n", dem_small[1, 1]))
cat(sprintf("  NE [1, ncol]:      %.1f m\n", dem_small[1, ncol(dem_small)]))
cat(sprintf("  SW [nrow, 1]:      %.1f m\n", dem_small[nrow(dem_small), 1]))
cat(sprintf("  SE [nrow, ncol]:   %.1f m\n", dem_small[nrow(dem_small), ncol(dem_small)]))
cat("  Centre [2:3, 2:3]:\n"); print(dem_small[2:3, 2:3])

# NIR band: [row, col, band]
cat(sprintf("\n  NIR band 3, centre [2, 2, 3]: %d\n", rgb_small[2, 2, 3]))

# ── 3. Vectorised math ────────────────────────────────────────────────────
dem_centred <- dem_small - mean(dem_small)
cat(sprintf("\nCentred (subtract mean %.1f m):\n", mean(dem_small)))
print(round(dem_centred, 1))

nir_r  <- as.numeric(rgb_small[,,3])
red_r  <- as.numeric(rgb_small[,,1])
ndvi_r <- (nir_r - red_r) / (nir_r + red_r)
cat(sprintf("\nNDVI range: [%.3f, %.3f]\n", min(ndvi_r), max(ndvi_r)))

# ── 4. Dtype cast ─────────────────────────────────────────────────────────
cat(sprintf("\nuint8 overflow (R integer): %dL + 100L stays numeric — R promotes to int\n",
    200L + 100L))
cat("In terra, use as.numeric() or terra::as.float() before arithmetic on integer rasters.\n")


🔧 **Try it yourself:** In the Python cell, change `dem_small[1:3, 1:3]` to `dem_small[0:2, 0:2]`. Which corner of the matrix are you now looking at? Confirm by checking the original printed values.

In the R cell, add a line: `cat(dem_small[1,1] > dem_small[nrow(dem_small),1], "\n")`. Is the NW corner higher or lower than the SW corner in this array?


---
## Section 2 — Loading Rasters: Four Libraries

This module uses three distinct raster types — each stored differently and requiring different statistical treatment:

| File | Dtype | Cell meaning | Statistical rule |
|---|---|---|---|
| `dem_30m.tif` | `float32` | Elevation in metres (continuous) | Mask NoData first, then mean/std |
| `nlcd_cover.tif` | `uint8` | NLCD class code (categorical) | Use frequency table only — never mean a class code |
| `landsat8.tif` | `uint16` | Scaled reflectance × 10 000, 7 bands | Cast to float32 before band math |

**Library map:**

| Library | Language | Role |
|---|---|---|
| `rasterio` | Python | GDAL-backed GeoTIFF I/O — the primary read/write tool |
| `numpy` | Python | Array arithmetic, masking, statistics |
| `terra` | R | Modern replacement for `raster` — fast, tidyverse-friendly |
| `stars` | R | Spatiotemporal array model — NetCDF, time series, multi-band native |
| `raster` | R | Legacy package — still in millions of online examples |


### 2a — Loading rasters in Python with `rasterio`


In [ ]:
# [Python] Loading all three raster types with rasterio
# rasterio.open() returns a DatasetReader — use it as a context manager

import rasterio
import numpy as np

paths = {
    "DEM (float32)":        "output/dem_30m.tif",
    "Land cover (uint8)":   "output/nlcd_cover.tif",
    "Landsat 8 (uint16)":  "output/landsat8.tif",
}

for label, path in paths.items():
    with rasterio.open(path) as src:
        arr = src.read()                   # read ALL bands → (bands, rows, cols)
        print(f"── {label}")
        print(f"   driver:  {src.driver}   dtype: {src.dtypes[0]}   bands: {src.count}")
        print(f"   shape:   {src.height} rows × {src.width} cols   NoData: {src.nodata}")
        print(f"   CRS:     {src.crs}")
        print(f"   array shape returned by read(): {arr.shape}")
        print()


### 2b — Loading rasters in R with `terra`, `stars`, and `raster`


In [ ]:
# [R] Loading rasters with terra (primary), stars, and the legacy raster package

library(terra)
library(stars)
library(raster)   # legacy — load to show differences

# ── terra: rast() returns a SpatRaster ────────────────────────────────────
dem    <- rast("output/dem_30m.tif")
nlcd   <- rast("output/nlcd_cover.tif")
ls8    <- rast("output/landsat8.tif")    # all 7 bands as a SpatRaster stack

cat("── terra SpatRaster: DEM ──\n");  print(dem)
cat("\n── terra SpatRaster: Landsat 8 ──\n"); print(ls8)

# ── stars: read_stars() returns a stars object ────────────────────────────
# stars is excellent for multi-band and time-series rasters; sf-compatible
dem_stars <- read_stars("output/dem_30m.tif")
cat("\n── stars object: DEM ──\n"); print(dem_stars)

# ── legacy raster package: raster() / stack() ─────────────────────────────
# You will encounter raster:: code in older scripts and Stack Overflow answers.
dem_r   <- raster("output/dem_30m.tif")    # single band → RasterLayer
ls8_stk <- stack("output/landsat8.tif")    # multi-band  → RasterStack
cat("\n── legacy raster() object ──\n"); print(dem_r)
cat(sprintf("\nRasterStack: %d layers\n", nlayers(ls8_stk)))

# Convert legacy to terra when you need to use modern functions:
dem_terra_from_raster <- rast(dem_r)
cat("\nConverted raster → terra SpatRaster: class =", class(dem_terra_from_raster), "\n")


🔧 **Try it yourself:** In the Python cell, load `output/nlcd_cover.tif` and print `src.dtypes`, `src.nodata`, and `src.count`. Then answer: why is `uint8` appropriate for NLCD category codes but NOT for elevation values?

In the R cell, after loading `nlcd <- rast("output/nlcd_cover.tif")`, run `datatype(nlcd)`. What does that string mean?


---
## Section 3 — The First-Inspection Checklist

Run this checklist on every raster before any analysis. It catches the majority of problems before they corrupt results silently.

| # | Property | Why it matters |
|---|---|---|
| 1 | **CRS** | Must match every other layer before any spatial operation |
| 2 | **Extent** | Is the bounding box geographically sensible? |
| 3 | **Resolution** | Cell size in CRS units — 30 m in degrees is a red flag |
| 4 | **Shape** | Rows × cols — is the file the expected size? |
| 5 | **Dtype** | float32 vs uint8 — determines what arithmetic is safe |
| 6 | **NoData** | What sentinel means "no data"? Must be masked before stats |
| 7 | **Value range** | Min/max after masking — does it make physical sense? |

> **Red flags to watch for:**  
> `Min: −9999.0` → NoData not yet masked  
> `Mean: 32.7` on a land cover raster → class codes averaged (meaningless)  
> `res: (0.000269, 0.000269)` on a "30 m" DEM → CRS is geographic degrees, not projected metres


### 3a — Checklist in Python


In [ ]:
# [Python] The seven-point first-inspection checklist — DEM

import rasterio
import numpy as np

with rasterio.open("output/dem_30m.tif") as src:
    print("=== DEM FIRST-INSPECTION CHECKLIST ===")
    print(f"1. CRS:          {src.crs}")
    print(f"   EPSG:         {src.crs.to_epsg()}")
    print(f"2. Extent:       {src.bounds}")
    print(f"3. Resolution:   x={src.res[0]:.1f} m   y={src.res[1]:.1f} m")
    print(f"4. Shape:        {src.height} rows × {src.width} cols")
    print(f"5. Dtype:        {src.dtypes[0]}")
    print(f"   Bands:        {src.count}")
    print(f"6. NoData:       {src.nodata}")
    print(f"   Driver:       {src.driver}")
    print(f"   Transform:\n  {src.transform}")

    # Read band 1, cast to float32, mask NoData
    arr = src.read(1).astype("float32")
    nd  = src.nodata
    arr[arr == nd] = np.nan                       # ← mask BEFORE any statistics

valid      = arr[~np.isnan(arr)]
n_nodata   = int(np.isnan(arr).sum())
pct_nodata = n_nodata / arr.size * 100

print(f"\n7. Value range (after NoData mask):")
print(f"   Min:          {np.min(valid):.1f} m")
print(f"   Max:          {np.max(valid):.1f} m")
print(f"   Mean:         {np.mean(valid):.1f} m")
print(f"   Std dev:      {np.std(valid):.1f} m")
print(f"   NoData cells: {n_nodata:,} of {arr.size:,} ({pct_nodata:.2f}%)")

# ── Demonstrate why masking is non-optional ────────────────────────────────
with rasterio.open("output/dem_30m.tif") as src:
    arr_raw = src.read(1).astype("float32")         # NO masking
print(f"\n⚠  WITHOUT masking:  min={np.min(arr_raw):.1f}  mean={np.mean(arr_raw):.1f}")
print(f"✅  WITH masking:     min={np.min(valid):.1f}  mean={np.mean(valid):.1f}")


### 3b — Checklist in R


In [ ]:
# [R] The seven-point first-inspection checklist with terra

library(terra)

r <- rast("output/dem_30m.tif")

cat("=== DEM FIRST-INSPECTION CHECKLIST ===\n")
cat(sprintf("1. CRS:          %s\n",  crs(r, describe = TRUE)$name))
cat(sprintf("   EPSG:         %s\n",  crs(r, describe = TRUE)$code))
cat(sprintf("2. Extent:       %s\n",  as.character(ext(r))))
cat(sprintf("3. Resolution:   x=%.1f m  y=%.1f m\n", res(r)[1], res(r)[2]))
cat(sprintf("4. Shape:        %d rows × %d cols\n", nrow(r), ncol(r)))
cat(sprintf("5. Dtype:        %s\n",  datatype(r)))
cat(sprintf("   Bands:        %d\n",  nlyr(r)))
cat(sprintf("6. NoData flag:  %s\n",  NAflag(r)))

# terra returns NA where NoData exists — na.rm = TRUE handles masking automatically
vals     <- values(r, na.rm = TRUE)
na_count <- as.integer(global(r, "isNA")$isNA)
pct_na   <- round(na_count / ncell(r) * 100, 2)

cat("\n7. Value range (after NoData mask):\n")
cat(sprintf("   Min:          %.1f m\n", min(vals)))
cat(sprintf("   Max:          %.1f m\n", max(vals)))
cat(sprintf("   Mean:         %.1f m\n", mean(vals)))
cat(sprintf("   Std dev:      %.1f m\n", sd(vals)))
cat(sprintf("   NoData cells: %d of %d (%.2f%%)\n",
    na_count, ncell(r), pct_na))

# terra shortcut: summary() gives min/q1/mean/q3/max in one call
cat("\nterra summary() shortcut:\n")
print(summary(r))


### 3c — Run the checklist on all three rasters


In [ ]:
# [Python] Loop the checklist across all three rasters — a quick sanity-check pattern

import rasterio
import numpy as np

files = {
    "DEM":        "output/dem_30m.tif",
    "NLCD":       "output/nlcd_cover.tif",
    "Landsat 8":  "output/landsat8.tif",
}

print(f"{'Raster':<12} {'Dtype':<9} {'Bands':>5} {'Shape':>12} {'NoData':>8} {'Min':>10} {'Max':>10} {'Mean':>10}")
print("─" * 80)

for name, path in files.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")   # inspect band 1
        nd  = src.nodata
        arr[arr == nd] = np.nan
        valid = arr[~np.isnan(arr)]
        shape = f"{src.height}×{src.width}"
        print(
            f"{name:<12} {src.dtypes[0]:<9} {src.count:>5} "
            f"{shape:>12} {str(nd):>8} "
            f"{np.min(valid):>10.1f} {np.max(valid):>10.1f} {np.mean(valid):>10.1f}"
        )

print("\nNote: NLCD mean (≈45.3) is a class code average — it refers to no real land class.")
print("      Never average categorical raster values.")


In [ ]:
# [R] Loop the checklist across all three rasters with terra

library(terra)

files <- list(
    DEM      = "output/dem_30m.tif",
    NLCD     = "output/nlcd_cover.tif",
    Landsat8 = "output/landsat8.tif"
)

cat(sprintf("%-12s %-9s %5s %14s %8s %10s %10s %10s\n",
    "Raster", "Dtype", "Bands", "Shape", "NoData", "Min", "Max", "Mean"))
cat(strrep("─", 78), "\n")

for (nm in names(files)) {
    r    <- rast(files[[nm]])
    vals <- values(r[[1]], na.rm = TRUE)   # band 1 for summary
    cat(sprintf("%-12s %-9s %5d %7d×%-6d %8s %10.1f %10.1f %10.1f\n",
        nm, datatype(r), nlyr(r), nrow(r), ncol(r),
        as.character(NAflag(r)), min(vals), max(vals), mean(vals)))
}
cat("\nNLCD mean ≈ 45.3 is meaningless — class codes are nominal, not continuous.\n")


🔧 **Try it yourself:** Run the Python checklist on `output/landsat8.tif` with `band=5` (NIR). Then change to `band=4` (Red). Compare the two means. The NIR mean should be roughly 5× the Red mean — a signature of healthy vegetation. Is that what you see?


---
## Section 4 — Indexing, Cell Extraction, and Windowed Reads

There are two ways to pull values out of a raster:

1. **Array indexing** — `arr[row, col]` or `m[row, col]`: use when you know pixel coordinates
2. **Windowed reads** — load a geographic tile without reading the whole file: essential for large rasters

> **Why windowed reads matter:** A single Sentinel-2 tile covering Washington state can be several gigabytes. You cannot read it fully into RAM. `rasterio.Window` and `terra::crop()` let you work on the tile you actually need.


### 4a — Array indexing in Python


In [ ]:
# [Python] Direct array indexing into the DEM — corners, centre, windows

import rasterio
import numpy as np

with rasterio.open("output/dem_30m.tif") as src:
    arr       = src.read(1).astype("float32")
    nd        = src.nodata
    transform = src.transform          # affine: pixel → geographic coords
    arr[arr == nd] = np.nan

rows, cols = arr.shape

# ── Corners and centre ────────────────────────────────────────────────────
print(f"Shape: {rows} rows × {cols} cols")
print(f"\nCorner values:")
print(f"  NW [0,    0   ]: {arr[0,    0   ]:.1f} m")
print(f"  NE [0,    cols]: {arr[0,    cols-1]:.1f} m  ← NaN if inside NoData hole")
print(f"  SW [rows, 0   ]: {arr[rows-1, 0  ]:.1f} m")
print(f"  SE [rows, cols]: {arr[rows-1, cols-1]:.1f} m")

cr, cc = rows // 2, cols // 2
print(f"  Centre [{cr}, {cc}]: {arr[cr, cc]:.1f} m")

# ── 10×10 window around centre ────────────────────────────────────────────
window_10 = arr[cr-5:cr+5, cc-5:cc+5]
print(f"\n10×10 window around centre (m):")
print(np.round(window_10, 0).astype(int))

# ── Convert pixel coordinates → geographic coords (affine transform) ──────
# rasterio.transform.xy(transform, row, col) → (x_easting, y_northing)
x, y = rasterio.transform.xy(transform, cr, cc)
print(f"\nCentre pixel geographic position: ({x:.1f} E, {y:.1f} N)  [EPSG:32610]")

# ── Count NoData cells ────────────────────────────────────────────────────
n_nd = int(np.isnan(arr).sum())
print(f"\nNoData cells: {n_nd:,}  ({n_nd/arr.size*100:.2f}% of {arr.size:,} total)")


### 4b — Windowed reads in Python


In [ ]:
# [Python] Windowed reads — load a tile without reading the whole raster

import rasterio
from rasterio.windows import Window, from_bounds
import numpy as np

# ── Approach 1: Window by pixel offsets ──────────────────────────────────
# Window(col_off, row_off, width, height)  — note: col first, then row
win_nw = Window(col_off=0,   row_off=0,   width=100, height=100)   # NW 100×100 tile
win_ctr = Window(col_off=117, row_off=117, width=100, height=100)  # centre tile

with rasterio.open("output/dem_30m.tif") as src:
    patch_nw  = src.read(1, window=win_nw).astype("float32")
    patch_ctr = src.read(1, window=win_ctr).astype("float32")
    nd        = src.nodata

    # Each window has its own affine transform — tells you where it is on Earth
    ctr_transform = src.window_transform(win_ctr)

    patch_nw [patch_nw  == nd] = np.nan
    patch_ctr[patch_ctr == nd] = np.nan

print(f"NW tile (100×100 px):")
print(f"  shape: {patch_nw.shape}  |  mean: {np.nanmean(patch_nw):.1f} m  "
      f"|  NW corner: {patch_nw[0,0]:.1f} m  "
      f"(NaN={np.isnan(patch_nw[0,0])}  ← inside NoData hole?)")

print(f"\nCentre tile (100×100 px):")
print(f"  shape: {patch_ctr.shape}  |  mean: {np.nanmean(patch_ctr):.1f} m")
print(f"  NW corner of tile, geographic: {ctr_transform * (0, 0)}")

# ── Approach 2: Window from geographic bounds ─────────────────────────────
# Use when you know your study-area extent in map coordinates, not pixels
with rasterio.open("output/dem_30m.tif") as src:
    xmin = src.bounds.left
    ymin = src.bounds.bottom
    # SW 3 km × 3 km tile
    win_sw = from_bounds(xmin, ymin, xmin + 3000, ymin + 3000, src.transform)
    patch_sw = src.read(1, window=win_sw).astype("float32")

print(f"\nSW 3 km × 3 km tile:  {patch_sw.shape[0]} rows × {patch_sw.shape[1]} cols")
print(f"  Elevation range: {patch_sw.min():.0f} – {patch_sw.max():.0f} m")


### 4c — Cell extraction and cropping in R


In [ ]:
# [R] Cell extraction and geographic cropping with terra

library(terra)

r  <- rast("output/dem_30m.tif")
nr <- nrow(r); nc <- ncol(r)
m  <- as.matrix(r, wide = TRUE)   # terra converts NA where NoData exists

cat("Shape:", nr, "rows ×", nc, "cols\n")

# ── Corners and centre ────────────────────────────────────────────────────
cat("\nCorner values (R is 1-indexed; row 1 = NORTH):\n")
cat(sprintf("  NW [1,  1 ]:       %.1f m\n", m[1,  1]))
cat(sprintf("  NE [1,  nc]:       %.1f m\n", m[1,  nc]))
cat(sprintf("  SW [nr, 1 ]:       %.1f m\n", m[nr, 1]))
cat(sprintf("  SE [nr, nc]:       %.1f m\n", m[nr, nc]))
cr <- nr %/% 2; cc <- nc %/% 2
cat(sprintf("  Centre [%d, %d]:   %.1f m\n", cr, cc, m[cr, cc]))

# ── 10×10 window around centre ────────────────────────────────────────────
cat("\n10×10 window around centre (m):\n")
print(round(m[(cr-5):(cr+4), (cc-5):(cc+4)], 0))

# ── Geographic crop: terra::crop() takes an Extent object ─────────────────
xmin_r <- xmin(ext(r)); ymin_r <- ymin(ext(r))
study_ext <- ext(xmin_r, xmin_r + 3000, ymin_r, ymin_r + 3000)
sub_r     <- crop(r, study_ext)

vals_sub  <- values(sub_r, na.rm = TRUE)
cat(sprintf("\nSW 3 km × 3 km crop: %d rows × %d cols\n", nrow(sub_r), ncol(sub_r)))
cat(sprintf("  Elevation range: %.0f – %.0f m\n", min(vals_sub), max(vals_sub)))

# ── Count NoData cells ────────────────────────────────────────────────────
na_n <- as.integer(global(r, "isNA")$isNA)
cat(sprintf("\nNoData cells: %d  (%.2f%% of %d total)\n",
    na_n, na_n / ncell(r) * 100, ncell(r)))


🔧 **Try it yourself:** In the Python windowed-read cell, change `win_nw` to load the **NE 100×100 tile** (that is, the top-right corner). What elevation values do you see? Are any cells NaN? Why? (Hint: look at the NoData hole created in the setup cell.)


---
## Section 5 — Summary Statistics: The Right Way for Each Raster Type

**The rule before every calculation: mask NoData first.**

Statistics differ fundamentally by raster type:

| Type | Useful statistics | What to avoid |
|---|---|---|
| Continuous (DEM, temperature) | min, max, mean, std, percentiles | Nothing — all make sense |
| Categorical (NLCD class codes) | frequency table, dominant class, count per class | mean, std — class codes are nominal |
| Multi-band (Landsat) | per-band stats, band ratios (NDVI, NDWI) | Cross-band mean without purpose |


### 5a — Continuous raster statistics (DEM)


In [ ]:
# [Python] Summary statistics for the DEM — continuous float32

import rasterio
import numpy as np

with rasterio.open("output/dem_30m.tif") as src:
    arr = src.read(1).astype("float32")
    arr[arr == src.nodata] = np.nan    # MASK FIRST

valid = arr[~np.isnan(arr)]

print("=== DEM Summary Statistics (elevation, metres) ===")
print(f"Valid pixels:    {len(valid):,} of {arr.size:,}")
print(f"Min:             {np.min(valid):.1f} m")
print(f"Max:             {np.max(valid):.1f} m")
print(f"Range:           {np.max(valid) - np.min(valid):.1f} m")
print(f"Mean:            {np.mean(valid):.1f} m")
print(f"Median:          {np.median(valid):.1f} m")
print(f"Std dev:         {np.std(valid):.1f} m")
print(f"10th pctile:     {np.percentile(valid, 10):.1f} m")
print(f"90th pctile:     {np.percentile(valid, 90):.1f} m")
print(f"IQR:             {np.percentile(valid, 75) - np.percentile(valid, 25):.1f} m")


### 5b — Categorical raster statistics (NLCD)


In [ ]:
# [Python] Frequency table for NLCD — the correct way to summarise a categorical raster

import rasterio
import numpy as np

NLCD_LABELS = {
    11: "Open Water",
    21: "Dev/Open Space",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Hay/Pasture",
    90: "Woody Wetlands",
}

with rasterio.open("output/nlcd_cover.tif") as src:
    arr = src.read(1)         # uint8 — leave as integer; do NOT cast to float for freq table
    nd  = int(src.nodata)

valid = arr[arr != nd]        # exclude background (0 = NoData)
codes, counts = np.unique(valid, return_counts=True)
total = counts.sum()

print(f"=== NLCD Land Cover Frequency Table ===")
print(f"{'Code':>5}  {'Label':<25}  {'Count':>8}  {'Pct':>6}")
print("─" * 50)
for code, count in sorted(zip(codes, counts), key=lambda x: -x[1]):
    label = NLCD_LABELS.get(int(code), "Unknown")
    print(f"{code:>5}  {label:<25}  {count:>8,}  {count/total*100:>5.1f}%")

dominant = codes[np.argmax(counts)]
print(f"\nDominant class: {dominant} — {NLCD_LABELS.get(int(dominant), '?')}")

# ── Why you must NOT average class codes ──────────────────────────────────
print(f"\n⚠  Mean of class codes: {np.mean(valid):.1f} — refers to no actual land class.")
print("    Class codes are labels on a nominal scale.  Frequency table only.")


In [ ]:
# [R] Frequency table with terra — NLCD categorical raster

library(terra)

nlcd <- rast("output/nlcd_cover.tif")

nlcd_labels <- c(
    "11" = "Open Water", "21" = "Dev/Open Space",
    "41" = "Deciduous Forest", "42" = "Evergreen Forest",
    "52" = "Shrub/Scrub", "71" = "Grassland",
    "81" = "Hay/Pasture", "90" = "Woody Wetlands"
)

# terra::freq() gives class frequency directly — handles NoData automatically
ftbl <- freq(nlcd, bylyr = FALSE)
ftbl <- ftbl[ftbl$value != 0, ]    # drop background (0 = NoData)
ftbl$label <- nlcd_labels[as.character(ftbl$value)]
ftbl$pct   <- round(ftbl$count / sum(ftbl$count) * 100, 1)
ftbl <- ftbl[order(-ftbl$count), ]

cat("=== NLCD Land Cover Frequency Table ===\n")
print(ftbl[, c("value", "label", "count", "pct")])

cat(sprintf("\nDominant class: %s — %s\n",
    ftbl$value[1], ftbl$label[1]))

# Show why mean is wrong
vals_all <- values(nlcd, na.rm = TRUE)
cat(sprintf("\nMean of codes: %.1f — meaningless for nominal data.\n", mean(vals_all)))


### 5c — Per-band statistics for Landsat 8


In [ ]:
# [Python] Per-band statistics for Landsat 8 — and NDVI

import rasterio
import numpy as np

BAND_NAMES = ["B1 Coastal", "B2 Blue", "B3 Green", "B4 Red",
              "B5 NIR", "B6 SWIR1", "B7 SWIR2"]

with rasterio.open("output/landsat8.tif") as src:
    nd = src.nodata
    print(f"=== Landsat 8 Per-Band Statistics (reflectance × 10 000) ===")
    print(f"{'Band':<12} {'Min':>6} {'Max':>6} {'Mean':>7} {'Std':>7}")
    print("─" * 44)
    bands = {}
    for i in range(1, src.count + 1):
        b = src.read(i).astype("float32")
        b[b == nd] = np.nan
        valid = b[~np.isnan(b)]
        bands[i] = b
        print(f"{BAND_NAMES[i-1]:<12} {int(np.min(valid)):>6} {int(np.max(valid)):>6} "
              f"{np.mean(valid):>7.0f} {np.std(valid):>7.0f}")

# ── NDVI: (NIR − Red) / (NIR + Red) ──────────────────────────────────────
nir  = bands[5]     # Band 5 = NIR
red  = bands[4]     # Band 4 = Red
denom = nir + red
ndvi  = np.where(denom != 0, (nir - red) / denom, np.nan)
valid_ndvi = ndvi[~np.isnan(ndvi)]
print(f"\nNDVI  (B5 NIR − B4 Red) / (B5 + B4):")
print(f"  Min:    {np.min(valid_ndvi):.3f}")
print(f"  Max:    {np.max(valid_ndvi):.3f}")
print(f"  Mean:   {np.mean(valid_ndvi):.3f}")
print(f"  Interpretation: values 0.3 – 0.8 indicate moderate to dense vegetation")


In [ ]:
# [R] Per-band statistics with terra + NDVI

library(terra)

ls8        <- rast("output/landsat8.tif")
band_names <- c("B1 Coastal","B2 Blue","B3 Green","B4 Red","B5 NIR","B6 SWIR1","B7 SWIR2")
names(ls8) <- band_names

cat("=== Landsat 8 Per-Band Statistics ===\n")
cat(sprintf("%-12s %6s %6s %7s %7s\n", "Band", "Min", "Max", "Mean", "Std"))
cat(strrep("─", 42), "\n")
for (i in 1:nlyr(ls8)) {
    v <- values(ls8[[i]], na.rm = TRUE)
    cat(sprintf("%-12s %6d %6d %7.0f %7.0f\n",
        band_names[i], min(v), max(v), mean(v), sd(v)))
}

# NDVI with terra — lapp() applies a function across layers element-wise
ndvi_r <- lapp(ls8[[c("B5 NIR", "B4 Red")]],
               fun = function(nir, red) (nir - red) / (nir + red))
v_ndvi <- values(ndvi_r, na.rm = TRUE)
cat(sprintf("\nNDVI:  min=%.3f  max=%.3f  mean=%.3f\n",
    min(v_ndvi), max(v_ndvi), mean(v_ndvi)))


🔧 **Try it yourself:** Add a cell computing **NDWI** (Normalised Difference Water Index) from the DEM's companion rasters:  
`NDWI = (Green − NIR) / (Green + NIR)` using Landsat bands 3 (Green) and 5 (NIR).  
NDWI > 0 typically indicates open water. Do any pixels in the synthetic scene exceed 0?


---
## Section 6 — Visualisation: Map, Histogram, and RGB Composite

Visualisation is a quality-control step, not just presentation. **Plot before computing statistics.** A histogram with a spike at −9999 tells you the NoData mask failed. An image that appears solid grey tells you you loaded one band when you needed three.

**Visualisation toolkit:**

| Task | Python | R |
|---|---|---|
| Quick single-band map | `plt.imshow()` | `plot(rast)` |
| Histogram | `plt.hist()` | `hist(vals)` |
| RGB composite | `plt.imshow(rgb.transpose(1,2,0))` | `plotRGB(sat)` |
| Publication map | `matplotlib + rasterio.plot.show()` | `ggplot2 + geom_raster()` |


### 6a — Single-band map + histogram in Python


In [ ]:
# [Python] Single-band maps and histograms — DEM and NLCD side by side

import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path

Path("output").mkdir(exist_ok=True)

# ── Load and mask ──────────────────────────────────────────────────────────
with rasterio.open("output/dem_30m.tif") as src:
    dem_arr = src.read(1).astype("float32")
    dem_arr[dem_arr == src.nodata] = np.nan

with rasterio.open("output/nlcd_cover.tif") as src:
    nlcd_arr = src.read(1).astype("float32")
    nlcd_arr[nlcd_arr == src.nodata] = np.nan

# ── Figure: 3 panels ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Module 5 — Raster Visualisation QA", fontsize=13,
             fontweight="bold", color="#1a1535")

# Panel 1: DEM — continuous terrain colourmap
im1 = axes[0].imshow(dem_arr, cmap="terrain", aspect="equal")
axes[0].set_title("DEM — 30 m elevation (m)", fontsize=11)
axes[0].set_xlabel("Column  (West → East)")
axes[0].set_ylabel("Row  (North → South)")
plt.colorbar(im1, ax=axes[0], label="Elevation (m)", shrink=0.85)

# Panel 2: NLCD — DISCRETE colourmap for categorical data
# Never use a continuous ramp for class codes — it implies order that doesn't exist
NLCD_COLORS = ["#5475A8","#E8D1D1","#68AB5F","#1C6330",
               "#CCB879","#E2E2C1","#DBD83D","#BAD8EA"]
NLCD_CODES  = [11, 21, 41, 42, 52, 71, 81, 90]
NLCD_LABELS = ["Water","Dev","Decid","Evergr","Shrub","Grass","Pasture","Wetland"]
cmap_nlcd = mcolors.ListedColormap(NLCD_COLORS)
bnorm     = mcolors.BoundaryNorm([0,15,25,35,45,60,75,85,95], cmap_nlcd.N)
axes[1].imshow(nlcd_arr, cmap=cmap_nlcd, norm=bnorm, aspect="equal")
axes[1].set_title("NLCD Land Cover — categorical", fontsize=11)
axes[1].set_xlabel("Column")
patches = [Patch(color=c, label=f"{code}:{lb}")
           for c, code, lb in zip(NLCD_COLORS, NLCD_CODES, NLCD_LABELS)]
axes[1].legend(handles=patches, loc="lower right", fontsize=6, ncol=2)

# Panel 3: DEM histogram — the sanity-check plot
valid = dem_arr[~np.isnan(dem_arr)].ravel()
axes[2].hist(valid, bins=60, color="#4b2e83", alpha=0.85, edgecolor="none")
axes[2].axvline(np.mean(valid), color="#b7a57a", lw=2,
                label=f"Mean: {np.mean(valid):.0f} m")
axes[2].set_title("DEM Elevation Histogram", fontsize=11)
axes[2].set_xlabel("Elevation (m)")
axes[2].set_ylabel("Pixel count")
axes[2].legend(fontsize=9)
axes[2].spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("output/raster_qa_panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/raster_qa_panel.png")


### 6b — RGB and false-colour composites in Python


In [ ]:
# [Python] Multi-band RGB and false-colour composites from Landsat 8

import rasterio
import numpy as np
import matplotlib.pyplot as plt

def percentile_stretch(arr_2d, lo=2, hi=98):
    """Clip to (lo, hi) percentile and scale to [0, 1] for display."""
    p_lo = np.nanpercentile(arr_2d, lo)
    p_hi = np.nanpercentile(arr_2d, hi)
    return np.clip((arr_2d - p_lo) / (p_hi - p_lo), 0, 1).astype("float32")

with rasterio.open("output/landsat8.tif") as src:
    nd = src.nodata
    def load(n):
        b = src.read(n).astype("float32")
        b[b == nd] = np.nan
        return b
    b2, b3, b4, b5 = load(2), load(3), load(4), load(5)

# Landsat 8 true colour: Red=B4, Green=B3, Blue=B2
true_col  = np.dstack([percentile_stretch(b4),
                       percentile_stretch(b3),
                       percentile_stretch(b2)])

# False-colour infrared: Red=B5(NIR), Green=B4(Red), Blue=B3(Green)
# Vegetation appears bright red — high NIR reflectance
false_col = np.dstack([percentile_stretch(b5),
                       percentile_stretch(b4),
                       percentile_stretch(b3)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Landsat 8 Composites — 2% percentile stretch",
             fontsize=12, fontweight="bold", color="#1a1535")

axes[0].imshow(true_col)
axes[0].set_title("True Colour  (B4, B3, B2 = R, G, B)", fontsize=11)
axes[0].set_xlabel("Column  (West → East)")
axes[0].set_ylabel("Row  (North → South)")

axes[1].imshow(false_col)
axes[1].set_title("False Colour NIR  (B5, B4, B3 = R, G, B)", fontsize=11)
axes[1].set_xlabel("Column")
axes[1].text(0.03, 0.04, "Bright red = high NIR = dense vegetation",
             transform=axes[1].transAxes, fontsize=8.5, color="white",
             bbox=dict(boxstyle="round", facecolor="#1a1535", alpha=0.75))

for ax in axes:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.savefig("output/landsat_composites.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: output/landsat_composites.png")
print("\n💡 Why grey? Loading one band and passing it to imshow gives a greyscale image.")
print("   You need THREE bands stacked (H, W, 3) for colour.  Always check src.count.")


### 6c — Single-band map and histogram in R


In [ ]:
# [R] Single-band visualisation with terra and ggplot2

library(terra)
library(ggplot2)

dem  <- rast("output/dem_30m.tif")
nlcd <- rast("output/nlcd_cover.tif")

# ── terra::plot() — the fastest inspection tool ────────────────────────────
par(mfrow = c(1, 2))
plot(dem,
     col  = terrain.colors(256),
     main = "DEM — 30 m elevation (m)",
     axes = TRUE)
plot(nlcd,
     col  = c("#5475A8","#E8D1D1","#68AB5F","#1C6330",
              "#CCB879","#E2E2C1","#DBD83D","#BAD8EA"),
     main = "NLCD Land Cover — categorical",
     axes = TRUE)
par(mfrow = c(1, 1))

# ── Elevation histogram ────────────────────────────────────────────────────
dem_vals <- values(dem, na.rm = TRUE)
hist(dem_vals,
     breaks = 60, col = "#4b2e83", border = NA,
     main   = "DEM Elevation Histogram",
     xlab   = "Elevation (m)", ylab = "Pixel count")
abline(v = mean(dem_vals), col = "#b7a57a", lwd = 2)
legend("topleft", legend = sprintf("Mean: %.0f m", mean(dem_vals)),
       col = "#b7a57a", lwd = 2, bty = "n")

# ── ggplot2: publication-quality DEM map ──────────────────────────────────
dem_df       <- as.data.frame(dem, xy = TRUE, na.rm = TRUE)
names(dem_df)[3] <- "elevation"

p <- ggplot(dem_df, aes(x = x, y = y, fill = elevation)) +
    geom_raster() +
    scale_fill_gradientn(colours = terrain.colors(256), name = "Elevation (m)") +
    coord_equal() +
    labs(title    = "DEM — Mt. Rainier Foothills Study Area",
         subtitle = "EPSG:32610 · UTM Zone 10N · 30 m resolution",
         x = "Easting (m)", y = "Northing (m)") +
    theme_minimal(base_size = 11) +
    theme(plot.title    = element_text(face = "bold", colour = "#1a1535"),
          plot.subtitle = element_text(colour = "#6b6b8a"))
print(p)
ggsave("output/dem_ggplot.png", p, width = 7, height = 6, dpi = 150)
cat("Saved: output/dem_ggplot.png\n")


### 6d — RGB composite with terra


In [ ]:
# [R] RGB and false-colour composites with terra::plotRGB()

library(terra)

ls8 <- rast("output/landsat8.tif")
names(ls8) <- c("B1_Coastal","B2_Blue","B3_Green","B4_Red",
                "B5_NIR","B6_SWIR1","B7_SWIR2")

par(mfrow = c(1, 2))

# True colour: r=B4, g=B3, b=B2
plotRGB(ls8, r = 4, g = 3, b = 2,
        stretch = "lin",
        main    = "True Colour (B4, B3, B2)")

# False-colour infrared: r=B5, g=B4, b=B3
plotRGB(ls8, r = 5, g = 4, b = 3,
        stretch = "hist",
        main    = "False Colour NIR (B5, B4, B3)")

par(mfrow = c(1, 1))

cat("plotRGB() stretch options:\n")
cat("  'lin'  = linear min-max stretch (most mapping tasks)\n")
cat("  'hist' = histogram equalisation (enhances low-contrast scenes)\n")
cat("\nWhy is the true-colour image desaturated? Synthetic band values are\n")
cat("uniform — a real scene would show spatial variation in reflectance.\n")


🔧 **Try it yourself:** In the Python visualisation cell (6a), add a **fourth panel** showing the NDVI computed in Section 5c, using the `'RdYlGn'` colourmap (red = low NDVI, green = high NDVI). Add a colourbar. Does the spatial pattern of NDVI match the elevation gradient? Why?


---
## Section 7 — Building the `inspect_raster()` Pipeline

Everything from Sections 3–6 can be wrapped in one function. You'll call this at the top of every Module 6 and 7 workflow — it's your raster equivalent of the vector sanity-check from Module 2.

The function should:
- Accept a file path and optional band number
- Print the full seven-point checklist
- Optionally render a map + histogram
- Return a dict (Python) or named list (R) so downstream code can use the values programmatically


### 7a — `inspect_raster()` in Python


In [ ]:
# [Python] inspect_raster() — reusable first-inspection function
# Carries into Modules 6 and 7 as the entry point for every raster workflow.

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def inspect_raster(path, band=1, plot=True, lo_pct=2, hi_pct=98):
    """
    Run the Module 5 seven-point checklist on any raster file.

    Parameters
    ----------
    path    : str | Path — path to raster (GeoTIFF, IMG, NetCDF, …)
    band    : int        — 1-based band index to summarise (default 1)
    plot    : bool       — render map + histogram side by side (default True)
    lo_pct  : float      — lower percentile for map display stretch (default 2)
    hi_pct  : float      — upper percentile for map display stretch (default 98)

    Returns
    -------
    dict — all checklist values; pass to downstream functions
    """
    with rasterio.open(path) as src:
        info = {
            "path":      str(path),
            "driver":    src.driver,
            "crs":       str(src.crs),
            "epsg":      src.crs.to_epsg(),
            "extent":    src.bounds,
            "res_x":     src.res[0],
            "res_y":     src.res[1],
            "nrows":     src.height,
            "ncols":     src.width,
            "nbands":    src.count,
            "dtype":     src.dtypes[band - 1],
            "nodata":    src.nodata,
            "transform": src.transform,
        }
        arr = src.read(band).astype("float32")

    if info["nodata"] is not None:
        arr[arr == info["nodata"]] = np.nan

    valid  = arr[~np.isnan(arr)]
    n_nd   = int(np.isnan(arr).sum())

    info.update({
        "valid_min":    float(np.min(valid))   if len(valid) else float("nan"),
        "valid_max":    float(np.max(valid))   if len(valid) else float("nan"),
        "valid_mean":   float(np.mean(valid))  if len(valid) else float("nan"),
        "valid_std":    float(np.std(valid))   if len(valid) else float("nan"),
        "nodata_count": n_nd,
        "nodata_pct":   round(n_nd / arr.size * 100, 3),
    })

    stem = Path(path).stem
    print(f"══ inspect_raster: {stem} ══")
    print(f"  CRS:          {info['crs']}  (EPSG:{info['epsg']})")
    print(f"  Extent:       {info['extent']}")
    print(f"  Resolution:   {info['res_x']:.1f} × {info['res_y']:.1f}  (CRS units)")
    print(f"  Shape:        {info['nrows']} rows × {info['ncols']} cols")
    print(f"  Bands:        {info['nbands']}   |   inspecting band {band}")
    print(f"  Dtype:        {info['dtype']}")
    print(f"  NoData:       {info['nodata']}  ({info['nodata_count']:,} px, {info['nodata_pct']:.2f}%)")
    print(f"  Valid range:  {info['valid_min']:.2f} – {info['valid_max']:.2f}")
    print(f"  Mean / Std:   {info['valid_mean']:.2f} / {info['valid_std']:.2f}")
    print()

    if plot and len(valid) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        p_lo = np.nanpercentile(arr, lo_pct)
        p_hi = np.nanpercentile(arr, hi_pct)
        im   = axes[0].imshow(arr, cmap="terrain", vmin=p_lo, vmax=p_hi, aspect="equal")
        axes[0].set_title(f"{stem} — band {band}", fontsize=11)
        axes[0].set_xlabel("Column  (West → East)")
        axes[0].set_ylabel("Row  (North → South)")
        plt.colorbar(im, ax=axes[0], shrink=0.9)
        axes[1].hist(valid.ravel(), bins=60, color="#4b2e83", alpha=0.85, edgecolor="none")
        axes[1].axvline(info["valid_mean"], color="#b7a57a", lw=2,
                        label=f"Mean: {info['valid_mean']:.1f}")
        axes[1].set_title("Histogram (valid pixels)", fontsize=11)
        axes[1].set_xlabel("Value")
        axes[1].set_ylabel("Pixel count")
        axes[1].legend(fontsize=9)
        axes[1].spines[["top","right"]].set_visible(False)
        plt.suptitle(f"inspect_raster: {stem}", fontsize=12,
                     fontweight="bold", color="#1a1535")
        plt.tight_layout()
        plt.show()

    return info

# ── Run on all three rasters ──────────────────────────────────────────────
for fpath in ["output/dem_30m.tif", "output/nlcd_cover.tif", "output/landsat8.tif"]:
    result = inspect_raster(fpath, plot=True)
    print(f"  → returned dict with keys: {list(result.keys())}\n")


### 7b — `inspect_raster()` in R


In [ ]:
# [R] inspect_raster() — R equivalent using terra

library(terra)

inspect_raster_r <- function(path, band = 1L, plot = TRUE) {
    #' Seven-point raster inspection checklist.
    #'
    #' @param path  Character. Path to raster file.
    #' @param band  Integer.   Band to summarise (1-based). Default 1.
    #' @param plot  Logical.   Render map + histogram. Default TRUE.
    #' @return Invisible named list with all checklist values.

    r  <- rast(path)
    b  <- r[[band]]

    vals    <- values(b, na.rm = TRUE)
    na_n    <- as.integer(global(b, "isNA")$isNA)
    total_n <- ncell(b)
    crs_info <- crs(r, describe = TRUE)

    info <- list(
        path        = path,
        crs         = crs_info$name,
        epsg        = crs_info$code,
        extent      = as.character(ext(r)),
        res         = res(r),
        nrows       = nrow(r),
        ncols       = ncol(r),
        nbands      = nlyr(r),
        dtype       = datatype(r),
        nodata_flag = NAflag(r),
        valid_min   = min(vals),
        valid_max   = max(vals),
        valid_mean  = mean(vals),
        valid_sd    = sd(vals),
        nodata_n    = na_n,
        nodata_pct  = round(na_n / total_n * 100, 3)
    )

    stem <- tools::file_path_sans_ext(basename(path))
    cat(sprintf("══ inspect_raster_r: %s ══\n", stem))
    cat(sprintf("  CRS:          %s  (EPSG:%s)\n", info$crs, info$epsg))
    cat(sprintf("  Extent:       %s\n", info$extent))
    cat(sprintf("  Resolution:   %.1f × %.1f  (CRS units)\n", info$res[1], info$res[2]))
    cat(sprintf("  Shape:        %d rows × %d cols\n", info$nrows, info$ncols))
    cat(sprintf("  Bands:        %d   |   inspecting band %d\n", info$nbands, band))
    cat(sprintf("  Dtype:        %s\n", info$dtype))
    cat(sprintf("  NoData flag:  %s  (%d px, %.2f%%)\n",
        info$nodata_flag, info$nodata_n, info$nodata_pct))
    cat(sprintf("  Valid range:  %.2f – %.2f\n", info$valid_min, info$valid_max))
    cat(sprintf("  Mean / SD:    %.2f / %.2f\n\n", info$valid_mean, info$valid_sd))

    if (plot && length(vals) > 0) {
        par(mfrow = c(1, 2))
        plot(b, col = terrain.colors(256),
             main = paste(stem, "— band", band), axes = TRUE)
        hist(vals, breaks = 60, col = "#4b2e83", border = NA,
             main = "Histogram (valid pixels)",
             xlab = "Value", ylab = "Pixel count")
        abline(v = info$valid_mean, col = "#b7a57a", lwd = 2)
        par(mfrow = c(1, 1))
    }

    invisible(info)
}

# ── Run on all three rasters ──────────────────────────────────────────────
dem_info  <- inspect_raster_r("output/dem_30m.tif")
nlcd_info <- inspect_raster_r("output/nlcd_cover.tif")
ls8_info  <- inspect_raster_r("output/landsat8.tif", band = 5L)   # inspect NIR band
cat("Function returns a named list. Access values with: dem_info$valid_mean\n")
cat(sprintf("DEM mean elevation: %.1f m\n", dem_info$valid_mean))


🔧 **Try it yourself:** Call `inspect_raster("output/landsat8.tif", band=5, plot=True)` then call it again with `band=4`. Print a comparison:
```python
nir = inspect_raster("output/landsat8.tif", band=5, plot=False)
red = inspect_raster("output/landsat8.tif", band=4, plot=False)
print(f"NIR mean: {nir['valid_mean']:.0f}    Red mean: {red['valid_mean']:.0f}")
print(f"NIR/Red ratio: {nir['valid_mean']/red['valid_mean']:.1f}×  (expect ~5× for vegetation)")
```


---
## Section 8 — ArcPy: Raster Inspection in the ESRI Ecosystem

ArcPy provides `arcpy.Describe()` and `arcpy.GetRasterProperties_management()` to read the same metadata you've been extracting with rasterio and terra. The concepts are identical — only the API differs.

**When to use ArcPy for rasters:**

| Scenario | Use |
|---|---|
| Raster stored in a File Geodatabase raster catalog | `arcpy` — GDAL cannot write raster catalogs |
| Running ArcGIS Spatial Analyst tools (slope, hillshade, reclassify) | `arcpy.sa` |
| Reading raster metadata in a cloned ArcGIS Pro env | `arcpy.Describe()` — same result as rasterio |
| Reading a plain GeoTIFF for analysis | `rasterio` — faster and portable |

> ⚠️ **ArcPy cells require a licensed ArcGIS Pro installation.** They will not run on JupyterHub. Run them inside the ArcGIS Pro built-in notebook, or in a Python kernel cloned from the `arcgispro-py3` environment.


In [ ]:
# [Python / ArcPy — Local ArcGIS Pro only]
# Raster inspection with arcpy.Describe() and GetRasterProperties_management()
# This cell prints a placeholder when ArcPy is unavailable — expected on JupyterHub.

try:
    import arcpy

    raster_path = "output/dem_30m.tif"   # or any path your workspace can reach

    # ── arcpy.Describe() — structural metadata ────────────────────────────
    desc = arcpy.Describe(raster_path)
    print("=== arcpy.Describe() ===")
    print(f"  dataType:        {desc.dataType}")
    print(f"  pixelType:       {desc.pixelType}")        # U8, S16, F32 …
    print(f"  bandCount:       {desc.bandCount}")
    print(f"  meanCellWidth:   {desc.meanCellWidth}")
    print(f"  meanCellHeight:  {desc.meanCellHeight}")
    print(f"  spatialRef:      {desc.spatialReference.name}")
    print(f"  extent:          {desc.extent}")
    print(f"  noDataValue:     {desc.noDataValue}")

    # ── GetRasterProperties_management() — value statistics ───────────────
    def prop(raster, key):
        return arcpy.GetRasterProperties_management(raster, key).getOutput(0)

    print("\n=== arcpy.GetRasterProperties_management() ===")
    for key in ["MINIMUM","MAXIMUM","MEAN","STD","CELLSIZEX","CELLSIZEY",
                "COLUMNCOUNT","ROWCOUNT","BANDCOUNT","ANYNODATA"]:
        print(f"  {key:<14}: {prop(raster_path, key)}")

except ModuleNotFoundError:
    print("ArcPy not available in this environment — expected on JupyterHub.")
    print("Run this cell in a cloned arcgispro-py3 kernel or inside ArcGIS Pro Notebook.")
    print()
    print("ArcPy raster property map:")
    print("  arcpy.Describe().pixelType       ←→ rasterio src.dtypes[0]")
    print("  arcpy.Describe().bandCount        ←→ rasterio src.count")
    print("  arcpy.Describe().meanCellWidth    ←→ rasterio src.res[0]")
    print("  arcpy.Describe().spatialReference ←→ rasterio src.crs")
    print("  GetRasterProperties('MINIMUM')    ←→ np.nanmin(arr)")
    print("  GetRasterProperties('ANYNODATA')  ←→ src.nodata is not None")


---

## 🔍 Module 5 — Self-Check Questions

Work through these without looking at the cells above. Then verify by running code.

**Arrays and data types**
1. A raster array has shape `(4, 500, 500)`. What does each dimension represent? Write the Python slice to extract the NW corner cell from the third band.
2. You call `np.mean(arr)` on a `uint8` NLCD raster without masking NoData (value = 0). The result is 32.7. Give two separate reasons this number is wrong.
3. Write the one-line Python fix to safely cast a `uint8` array to `float32` before arithmetic.

**Metadata and loading**
4. List all seven properties in the Module 5 first-inspection checklist and briefly explain why each matters.
5. You open a raster and `src.crs.to_epsg()` returns `4326`. You expected 30 m cells but `src.res` returns `(0.000269, 0.000269)`. What is happening, and what is the fix?
6. In R, what function do you use to load a 7-band GeoTIFF as a single object with all bands accessible? What property tells you how many bands loaded?

**Indexing and windowed reads**
7. Why does `arr[0, 0]` refer to the NW corner of the raster, not the SW corner? When would `arr[0, 0]` and the geographic NW corner NOT correspond?
8. You have a 12 GB Sentinel-2 raster. Write the two Python lines to load only the NW 256×256 pixel tile without reading the full file.
9. In terra (R), what function crops a SpatRaster to a geographic extent? Write the call to crop to a bounding box defined by `xmin, xmax, ymin, ymax` values.

**Statistics and visualisation**
10. A DEM histogram shows a sharp spike at −9999 and a bell curve between 300–1200 m. Diagnose the problem and write the fix.
11. You plot a Landsat RGB composite and the image is solid grey. Name two causes and explain how to diagnose each.
12. What NDVI value range indicates dense, healthy vegetation? What value range indicates bare soil or water?


---
## ✅ Module 5 Homework — Deliverables

Submit the following in Canvas before the next session.

**Assessment:** Complete Jupyter Notebook demonstrating loading and describing several types of raster data.

---

### Deliverable 1 — Raster Type Round-Trip (25 pts)

Using the synthetic rasters from this notebook **or your own real rasters from agency work**:

- Load one **continuous** raster (DEM or temperature surface) and one **categorical** raster (land cover or classified image)
- Call `inspect_raster()` on both; capture the printed output in your notebook
- Write a markdown cell (3–5 sentences) interpreting the results: does the value range make geographic sense? Is the CRS correct for your study area? Are there NoData issues to investigate?

Submit: screenshot of both `inspect_raster()` outputs **and** your written interpretation.

---

### Deliverable 2 — Statistics and Red Flags (25 pts)

For the DEM (`output/dem_30m.tif` or your own):

1. Compute the full statistics checklist **with** and **without** NoData masking in a single code cell; print both side by side
2. Produce a 2-panel histogram: raw array (NoData included) in the left panel, masked valid pixels in the right panel — label both axes and add a title to each panel
3. In a markdown cell, answer: how many cells are NoData, what percentage of the raster is that, and what real-world phenomenon could cause a similar hole in a real DEM?

Submit: the code cell + both-panel histogram figure + the markdown answer.

---

### Deliverable 3 — Multi-band Visualisation and Band Indices (50 pts)

Using `output/landsat8.tif` or a real Landsat/Sentinel scene:

1. Produce a **3-panel figure** saved as a PNG:
   - Panel 1: True-colour RGB composite (B4, B3, B2)
   - Panel 2: False-colour NIR composite (B5, B4, B3)
   - Panel 3: NDVI map using the `'RdYlGn'` colourmap with a colourbar
2. Print the per-band statistics table for all 7 bands
3. In a markdown cell, answer:
   - Which band has the highest mean reflectance, and why does that make physical sense?
   - What spatial pattern does the NDVI map show relative to the DEM elevation gradient? Why?
   - If this were a real Landsat scene, what pre-processing steps would you need before computing NDVI? (Hint: atmospheric correction, scale factor)

Submit: the 3-panel PNG, the statistics table output, and the markdown answers. Commit the notebook and PNG to your GitHub repo and tag the commit `module5-deliverable`.

---

**Stuck?** Post in Ed Discussion — tag `#module5`. Include your error message, the cell that failed, and your OS/environment. If you solved something tricky, share how — the whole cohort benefits.

**Looking ahead:** Module 6 (Raster Operations) uses `inspect_raster()` as its entry point for every workflow. If your DEM shows a wrong CRS or suspicious NoData percentage here, fix it before the next session.
